# 7-Day Window V3: LightGBM vs XGBoost
This notebook trains LGBM and XGB on the `LGBM_XGB_7_V3` dataset, compares MAE/SMAPE/BIAS/RMSE, and visualizes predictions and residuals.

## 1) Imports and Paths

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error

from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

RANDOM_SEED = 42
ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data' / 'processed' / 'LGBM_XGB_7_V3'
REPORTS_DIR = ROOT / 'artifacts' / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR:', DATA_DIR)
print('REPORTS_DIR:', REPORTS_DIR)

## 2) Load Train/Val/Test

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

for df in [train_df, val_df, test_df]:
    df['date'] = pd.to_datetime(df['date'], errors='coerce')

summary = pd.DataFrame([
    {'split': 'train', 'rows': len(train_df), 'cols': train_df.shape[1], 'null_cells': int(train_df.isna().sum().sum())},
    {'split': 'val',   'rows': len(val_df),   'cols': val_df.shape[1],   'null_cells': int(val_df.isna().sum().sum())},
    {'split': 'test',  'rows': len(test_df),  'cols': test_df.shape[1],  'null_cells': int(test_df.isna().sum().sum())},
])
summary

## 3) Features and Target

In [ ]:
target_col = 'aggregated_sales_7'

feature_cols = [c for c in train_df.columns if c not in [
    target_col, 'date', 'item_id'
]]

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()
X_val   = val_df[feature_cols].copy()
y_val   = val_df[target_col].copy()
X_test  = test_df[feature_cols].copy()
y_test  = test_df[target_col].copy()

categorical_features = [c for c in feature_cols if not pd.api.types.is_numeric_dtype(X_train[c])]
numeric_features     = [c for c in feature_cols if pd.api.types.is_numeric_dtype(X_train[c])]

print('Target:', target_col)
print('Feature count:', len(feature_cols))
print('Numeric:', len(numeric_features), '| Categorical:', len(categorical_features))
print('Features:', feature_cols)

## 4) Metric Helpers and Pipelines

In [ ]:
def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return float(100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / denom))

def compute_metrics(y_true, y_pred):
    return {
        'MAE':   float(mean_absolute_error(y_true, y_pred)),
        'RMSE':  float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'SMAPE': float(smape(y_true, y_pred)),
        'BIAS':  float(np.mean(np.asarray(y_pred) - np.asarray(y_true))),
    }

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

lgbm_model = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', LGBMRegressor(
        random_state=RANDOM_SEED,
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
    ))
])

xgb_model = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', XGBRegressor(
        random_state=RANDOM_SEED,
        n_estimators=400,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='reg:squarederror',
        tree_method='hist',
    ))
])

## 5) Train Both Models and Compare Metrics

In [ ]:
models = {
    'LightGBM_7d': lgbm_model,
    'XGBoost_7d':  xgb_model,
}

preds_val  = {}
preds_test = {}
rows = []

for name, model in models.items():
    model.fit(X_train, y_train)

    p_val  = model.predict(X_val)
    p_test = model.predict(X_test)

    preds_val[name]  = p_val
    preds_test[name] = p_test

    m_val  = compute_metrics(y_val,  p_val)
    m_test = compute_metrics(y_test, p_test)

    rows.append({'split': 'val',  'model': name, **m_val})
    rows.append({'split': 'test', 'model': name, **m_test})
    print(name, 'done')

metrics_df = pd.DataFrame(rows).sort_values(['split', 'MAE']).reset_index(drop=True)
metrics_df

## 6) Actual vs Predicted (Test)

In [ ]:
plot_df = test_df[['date']].copy()
plot_df['actual'] = y_test.values
for name, p in preds_test.items():
    plot_df[name] = p

daily = plot_df.groupby('date', as_index=False).mean(numeric_only=True).sort_values('date')

plt.figure(figsize=(12, 5))
plt.plot(daily['date'], daily['actual'],        label='Actual',       linewidth=2.5, color='black')
plt.plot(daily['date'], daily['LightGBM_7d'],   label='LightGBM_7d',  linewidth=2)
plt.plot(daily['date'], daily['XGBoost_7d'],    label='XGBoost_7d',   linewidth=2)
plt.title('Actual vs Predicted (Test, daily average)')
plt.xlabel('Date')
plt.ylabel('aggregated_sales_7')
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 7) Residual Distributions (Test)

In [ ]:
res_lgbm = preds_test['LightGBM_7d'] - y_test.values
res_xgb  = preds_test['XGBoost_7d']  - y_test.values

plt.figure(figsize=(12, 5))
plt.hist(res_lgbm, bins=80, alpha=0.55, density=True, label='LightGBM_7d residuals')
plt.hist(res_xgb,  bins=80, alpha=0.55, density=True, label='XGBoost_7d residuals')
plt.axvline(0, color='black', linestyle='--', linewidth=1)
plt.title('Residual Distributions on Test')
plt.xlabel('Residual (prediction - actual)')
plt.ylabel('Density')
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 8) Save Artifacts

In [ ]:
metrics_path = REPORTS_DIR / 'metrics_lgbm_xgb_7d_v3_comparison.csv'
metrics_df.to_csv(metrics_path, index=False)

preds_out = test_df[['item_id', 'date']].copy()
preds_out['actual']        = y_test.values
preds_out['pred_lgbm_7d']  = preds_test['LightGBM_7d']
preds_out['pred_xgb_7d']   = preds_test['XGBoost_7d']

preds_path = REPORTS_DIR / 'predictions_lgbm_xgb_7d_v3_test.csv'
preds_out.to_csv(preds_path, index=False)

print('Saved metrics:     ', metrics_path)
print('Saved predictions: ', preds_path)